## Jones and Muller matrices

In [1]:
import sympy as sp

In [18]:
def jones_to_mueller(J):
    """
    Compute Mueller matrix from a 2x2 Jones matrix using SymPy.

    Parameters
    ----------
    J : sympy.Matrix (2x2)
        Jones matrix (can be symbolic or numeric)

    Returns
    -------
    M : sympy.Matrix (4x4)
        Mueller matrix
    """

    # Hermitian conjugate
    Jd = J.conjugate().T

    # Stokes basis (I, Q, U, V)
    sigma = [
        sp.Matrix([[1, 0],
                   [0, 1]]),  # I

        sp.Matrix([[1, 0],
                   [0, -1]]), # Q

        sp.Matrix([[0, 1],
                   [1, 0]]),  # U

        sp.Matrix([[0, -sp.I],
                   [sp.I, 0]]) # V
    ]

    # Build Mueller matrix
    M = sp.zeros(4, 4)

    for i in range(4):
        for j in range(4):
            M[i, j] = sp.simplify(
                sp.Rational(1, 2) *
                (sigma[i] * J * sigma[j] * Jd).trace()
            )

    return M

def mueller_rotation(phi):
    """
    Mueller rotation matrix for Stokes parameters (Q,U) rotation by angle phi.
    
    Parameters
    ----------
    phi : sympy symbol or expression
        Rotation angle (in radians)

    Returns
    -------
    R : sympy.Matrix (4x4)
        Mueller rotation matrix
    """

    c = sp.cos(2*phi)
    s = sp.sin(2*phi)

    R = sp.Matrix([
        [1,  0,  0, 0],
        [0,  c,  s, 0],
        [0, -s,  c, 0],
        [0,  0,  0, 1]
    ])

    return sp.simplify(R)

### Linear polarizer

In [13]:
J_det = sp.Matrix([[1, 0],
               [0, 0]])

M_det = jones_to_mueller(J_det)

In [14]:
M_det

Matrix([
[1/2, 1/2, 0, 0],
[1/2, 1/2, 0, 0],
[  0,   0, 0, 0],
[  0,   0, 0, 0]])

In [16]:
J_hwp = sp.Matrix([[1, 0],
               [0, -1]])

H = jones_to_mueller(J_hwp)

In [17]:
H

Matrix([
[1, 0,  0,  0],
[0, 1,  0,  0],
[0, 0, -1,  0],
[0, 0,  0, -1]])

### Rotator

In [23]:
theta = sp.symbols('theta')
R = mueller_rotation(theta)

In [24]:
R

Matrix([
[1,             0,            0, 0],
[0,  cos(2*theta), sin(2*theta), 0],
[0, -sin(2*theta), cos(2*theta), 0],
[0,             0,            0, 1]])

In [27]:
varphi, alpha = sp.symbols('varphi_t alpha_t')

In [30]:
R_hwp = mueller_rotation(varphi)
R_det = mueller_rotation(alpha)

### Optical chain

In [90]:
H_rot = sp.trigsimp(R_hwp.T @ H @ R_hwp)
M = M_det @ H_rot @ R_det
M = sp.trigsimp(M)

In [91]:
M

Matrix([
[1/2, (cos(2*(alpha_t - 2*varphi_t)) + cos(2*(alpha_t + 2*varphi_t)))/4 - sin(2*alpha_t)*sin(4*varphi_t)/2, sin(2*alpha_t + 4*varphi_t)/2, 0],
[1/2, (cos(2*(alpha_t - 2*varphi_t)) + cos(2*(alpha_t + 2*varphi_t)))/4 - sin(2*alpha_t)*sin(4*varphi_t)/2, sin(2*alpha_t + 4*varphi_t)/2, 0],
[  0,                                                                                                    0,                             0, 0],
[  0,                                                                                                    0,                             0, 0]])